# 03 — Penggabungan Dataset HuggingFace + YouTube

Notebook ini menggabungkan dua sumber data emosi bahasa Indonesia:
1. **HuggingFace** — dataset publik `elvanromp/emosi`
2. **YouTube** — hasil labeling komentar YouTube (`dataset_emosi_single_label.csv`)

---

## Alur Notebook
1. **Install & Import** — library yang dibutuhkan
2. **Load Dataset HuggingFace** — download & bersihkan
3. **Load Dataset YouTube** — baca & sesuaikan format
4. **Handling Imbalance** — sampling Sadness YouTube
5. **Gabungkan** — train/val/test final
6. **Simpan** — `train_final.csv`, `val_final.csv`, `test_final.csv`

> **Output:** 3 file dataset siap training: `train_final.csv`, `val_final.csv`, `test_final.csv`

## 1. Install & Import

In [ ]:
!pip install datasets pandas -q

In [ ]:
import pandas as pd
from datasets import load_dataset

print('Import selesai')

## 2. Load Dataset HuggingFace

Dataset `elvanromp/emosi` adalah dataset klasifikasi emosi bahasa Indonesia yang tersedia di HuggingFace Hub.
Dataset ini sudah dibagi menjadi split `train`, `validation`, dan `test`.

In [ ]:
dataset = load_dataset('elvanromp/emosi')

train_df = dataset['train'].to_pandas()
val_df   = dataset['validation'].to_pandas()
test_df  = dataset['test'].to_pandas()

print(f'Train : {len(train_df)} baris')
print(f'Val   : {len(val_df)} baris')
print(f'Test  : {len(test_df)} baris')
print(f'Kolom : {train_df.columns.tolist()}')
train_df.head()

### 2.1 Bersihkan & Seragamkan Format

- Hapus label `surprise` agar konsisten dengan 7 emosi target
- Rename kolom: `text` → `normalize_text`, `label_text` → `emotion`
- Tambah kolom `source` untuk tracking asal data

In [ ]:
for df in [train_df, val_df, test_df]:
    # Hapus label 'surprise' (tidak ada di target 7 emosi)
    df.drop(df[df['label_text'] == 'surprise'].index, inplace=True)

    # Seragamkan nama kolom
    df.rename(columns={'text': 'normalize_text', 'label_text': 'emotion'}, inplace=True)

    # Tambah kolom sumber
    df['source'] = 'huggingface'

    # Hapus kolom label numerik (akan dibuat ulang di notebook 04)
    df.drop(columns=['label'], inplace=True)
    df.reset_index(drop=True, inplace=True)

print('Distribusi label HuggingFace (train):')
print(train_df['emotion'].value_counts())

## 3. Load Dataset YouTube

Dataset hasil labeling komentar YouTube dari notebook `02_labeling_emosi.ipynb`.

In [ ]:
df_yt = pd.read_csv('dataset_emosi_single_label.csv')

# Seragamkan nama kolom
df_yt.rename(columns={'label_single': 'emotion'}, inplace=True)
df_yt['emotion'] = df_yt['emotion'].str.lower()
df_yt['source']  = 'youtube'

print(f'Total data YouTube: {len(df_yt)} baris')
print()
print('Distribusi label YouTube (sebelum sampling):')
print(df_yt['emotion'].value_counts())
df_yt.head()

## 4. Handling Class Imbalance (Sampling Sadness)

Dataset YouTube didominasi oleh label `sadness` (~67%). Untuk mengurangi efek imbalance
saat digabung dengan HuggingFace, kita batasi sampel `sadness` menjadi **50 baris**.

In [ ]:
sadness_50  = df_yt[df_yt['emotion'] == 'sadness'].sample(n=50, random_state=42)
non_sadness = df_yt[df_yt['emotion'] != 'sadness']

df_yt_final = pd.concat([sadness_50, non_sadness], ignore_index=True)

print('Distribusi label YouTube (setelah sampling):')
print(df_yt_final['emotion'].value_counts())

## 5. Gabungkan Dataset

- **Train:** HuggingFace train + YouTube (sudah di-balance)
- **Validation:** HuggingFace validation saja
- **Test:** HuggingFace test saja

> YouTube tidak dimasukkan ke val/test untuk menjaga validitas evaluasi.

In [ ]:
train_final = pd.concat([train_df, df_yt_final], ignore_index=True)
val_final   = val_df.copy()
test_final  = test_df.copy()

print(f'Train final : {len(train_final)} baris')
print(f'Val final   : {len(val_final)} baris')
print(f'Test final  : {len(test_final)} baris')
print()
print('Distribusi emosi train final:')
print(train_final['emotion'].value_counts())
print()
print('Sumber data train:')
print(train_final['source'].value_counts())

## 6. Simpan Output

In [ ]:
train_final.to_csv('train_final.csv', index=False)
val_final.to_csv('val_final.csv',     index=False)
test_final.to_csv('test_final.csv',   index=False)

print('File tersimpan:')
print(f'  train_final.csv : {len(train_final)} baris')
print(f'  val_final.csv   : {len(val_final)} baris')
print(f'  test_final.csv  : {len(test_final)} baris')

## Ringkasan

| Item | Nilai |
|------|-------|
| Sumber 1 | HuggingFace `elvanromp/emosi` |
| Sumber 2 | YouTube `dataset_emosi_single_label.csv` |
| Label dihapus | `surprise` |
| Penanganan imbalance | Sampling Sadness YouTube → 50 baris |
| Output train | `train_final.csv` |
| Output val | `val_final.csv` |
| Output test | `test_final.csv` |